# Alytes-ReID: Setup & Training

This notebook handles:
1. Environment setup and dependency installation
2. Data download (iNaturalist + Roboflow)
3. YOLOv8 detection model training
4. SAM2 segmentation validation
5. Re-ID model training (when labeled data is available)
6. Model export to Google Drive

**Run this once** to prepare the models, then use `02_toad_reid.ipynb` for daily use.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danort92/Alytes-ReID/blob/main/notebooks/01_setup_and_training.ipynb)

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

# Clone repository
!git clone https://github.com/danort92/Alytes-ReID.git
%cd Alytes-ReID

# Install dependencies
!pip install -r requirements.txt -q

# Mount Google Drive for model storage
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/Alytes-ReID/models'
!mkdir -p {DRIVE_MODEL_DIR}

## 2. Data Download

In [ ]:
# Download Alytes images from iNaturalist
from src.data.download_inat import download_alytes_images
from pathlib import Path

inat_images = download_alytes_images(
    output_dir=Path('data/raw/inaturalist'),
    max_images=500,  # Start with 500, increase if needed
)
print(f'Downloaded {len(inat_images)} images from iNaturalist')

In [ ]:
# Optional: Download Roboflow dataset (needs API key)
# Uncomment and add your API key:

# from src.data.download_roboflow import download_roboflow_dataset
# download_roboflow_dataset(
#     output_dir=Path('data/raw/roboflow'),
#     api_key='YOUR_ROBOFLOW_API_KEY',
# )

## 3. Data Exploration

In [ ]:
import matplotlib.pyplot as plt
import cv2
import random

# Show sample images
sample = random.sample(inat_images, min(8, len(inat_images)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_path in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name[:20])
    ax.axis('off')
plt.suptitle('Sample iNaturalist Images')
plt.tight_layout()
plt.show()

## 4. Dataset Preparation

Prepare images and labels in YOLO format for training.

In [ ]:
# TODO: Annotation step
# For now, use Roboflow's pre-annotated data or annotate manually.
# The prepare_dataset script expects images/ and labels/ directories
# in YOLO format (class x_center y_center width height, normalized).

from src.data.prepare_dataset import split_dataset, create_yolo_dataset_yaml

# Example with Roboflow data (already in YOLO format):
# split_dataset(
#     images_dir=Path('data/raw/roboflow/train/images'),
#     labels_dir=Path('data/raw/roboflow/train/labels'),
#     output_dir=Path('data/processed/detection'),
# )
# create_yolo_dataset_yaml(
#     dataset_dir=Path('data/processed/detection'),
#     classes=['toad'],
# )

## 5. YOLOv8 Detection Training

In [ ]:
from src.detection.train import load_config, train_detector

config = load_config(Path('config/detection.yaml'))

# Train detector (requires prepared dataset from step 4)
# best_weights = train_detector(config)
# print(f'Best weights saved to: {best_weights}')

## 6. Evaluate Detection

In [ ]:
from src.detection.evaluate import evaluate_model

# metrics = evaluate_model(best_weights, config)
# print('Detection metrics:', metrics)

## 7. SAM2 Segmentation Test

In [ ]:
# Test SAM2 segmentation on a sample image with a detection bbox

# from src.segmentation.segment import ToadSegmenter
# from src.detection.predict import load_detector, detect_toads, get_best_detection
# from src.utils.visualization import draw_mask_overlay
#
# detector = load_detector(best_weights)
# segmenter = ToadSegmenter()
#
# test_image_path = inat_images[0]
# test_image = cv2.cvtColor(cv2.imread(str(test_image_path)), cv2.COLOR_BGR2RGB)
#
# detections = detect_toads(detector, test_image_path)
# best_det = get_best_detection(detections)
#
# if best_det:
#     cropped, mask = segmenter.segment_and_crop(test_image, best_det['bbox'])
#     overlay = draw_mask_overlay(test_image, 
#         segmenter.segment_from_bbox(test_image, best_det['bbox']))
#     
#     fig, axes = plt.subplots(1, 3, figsize=(15, 5))
#     axes[0].imshow(test_image); axes[0].set_title('Original')
#     axes[1].imshow(overlay); axes[1].set_title('Segmentation')
#     axes[2].imshow(cropped); axes[2].set_title('Cropped')
#     for ax in axes: ax.axis('off')
#     plt.show()

## 8. Re-ID Model Training

Requires labeled data with individual IDs (provided by the biologist).

In [ ]:
# from src.reid.train import train_reid
#
# reid_config = load_config(Path('config/reid.yaml'))
# model_path = train_reid(reid_config, data_dir=Path('data/processed/reid'))
# print(f'Re-ID model saved to: {model_path}')

## 9. Export Models to Google Drive

In [ ]:
import shutil

# Copy trained models to Google Drive for persistence
# shutil.copy2(str(best_weights), f'{DRIVE_MODEL_DIR}/detection_best.pt')
# shutil.copy2(str(model_path), f'{DRIVE_MODEL_DIR}/reid_model.pt')

print(f'Models would be saved to: {DRIVE_MODEL_DIR}')
print('Uncomment the copy lines above after training completes.')